# News Article Intelligence Pipeline

A data pipeline that scrapes news articles, performs statistical analysis with **pandas** and **numpy**, then automatically groups articles by topic using **scikit-learn** using kmeans clustering.

| Library | Role |
|---|---|
| `pandas` | organized articles into a dataframe|
| `numpy` | Vectorized stats and performed matrix operations |
| `scikit-learn` | Used kmeans to group articles together |
| `newspaper3k and BeautifulSoup` | Scrape html from articles|


Install Dependencies

In [10]:
import subprocess, sys
pkgs = ["newspaper3k", "requests", "beautifulsoup4", "pandas", "numpy", "scikit-learn", "lxml"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
print("All packages installed")

All packages installed


Imports

In [11]:
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

try:
    from newspaper import Article
    NEWSPAPER_AVAILABLE = True
except ImportError:
    NEWSPAPER_AVAILABLE = False
    print("newspaper3k unavailable — using BeautifulSoup fallback only.")

print("Imports complete")

newspaper3k unavailable — using BeautifulSoup fallback only.
Imports complete


Scraper
Fetches articles with `newspaper3k` (primary) and falls back to `BeautifulSoup` (secondary).  
Returns a `pandas.DataFrame`.


In [12]:
def _scrape_with_newspaper(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return {
            "title":  article.title,
            "author": ", ".join(article.authors) if article.authors else "Unknown",
            "text":   article.text,
            "url":    url,
        }
    except Exception as e:
        print(f"  [newspaper3k] {e}")
        return None


def _scrape_with_bs4(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, "html.parser")

        title_tag = soup.find("h1", class_="article-title") or soup.find("h1") or soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else "Untitled"

        content_div = soup.find("div", class_="article-content") or soup.find("article") or soup.find("main")
        if content_div:
            text = "\n".join(p.get_text(strip=True) for p in content_div.find_all("p") if p.get_text(strip=True))
        else:
            text = ""

        return {"title": title, "author": "Unknown", "text": text, "url": url}
    except Exception as e:
        print(f"  [bs4 fallback] {e}")
        return None


def scrape_single_article(url):
    result = _scrape_with_newspaper(url) if NEWSPAPER_AVAILABLE else None
    if result is None or not result.get("text"):
        result = _scrape_with_bs4(url)
    return result


def scrape_articles(urls):
    #Scrapes a list of URLs and give a dataframe
    records = []
    for i, url in enumerate(urls, 1):
        print(f"[{i}/{len(urls)}] {url}")
        result = scrape_single_article(url)
        if result and len(result.get("text", "")) > 200:
            records.append(result)
        else:
            print("  → Skipped (no content)")

    if not records:
        raise ValueError("No articles could be scraped.")

    df = pd.DataFrame(records)
    df["char_count"] = df["text"].str.len()
    df = df[df["char_count"] > 200].reset_index(drop=True)
    print(f"\n✓ {len(df)} usable articles scraped.")
    return df

Feature Engineering with pandas & numpy

Engineers interpretable text features on each article.


In [13]:
_STOPWORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "by","from","is","was","are","were","be","been","has","have","had",
    "do","does","did","will","would","could","should","may","might","not",
    "no","so","as","than","that","this","these","those","it","its",
    "he","she","they","we","you","i","me","him","her","us","them",
    "my","your","his","our","their","which","who","what","how","when","where",
}

def clean_text(text):
    #Lowercase, strip punctuation/digits/stopwords
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation + "0123456789"))
    tokens = [t for t in text.split() if t not in _STOPWORDS and len(t) > 2]
    return " ".join(tokens)


def compute_features(df):
    df = df.copy()
    df["clean_text"] = df["text"].apply(clean_text)

    raw_tokens = df["text"].str.lower().str.split()

    # pandas: word / sentence counts
    df["word_count"]      = raw_tokens.apply(len)
    df["sentence_count"]  = df["text"].apply(lambda t: max(1, len(re.findall(r"[.!?]+", t))))

    # numpy: mean / std of word lengths
    df["avg_word_length"] = raw_tokens.apply(
        lambda tok: float(np.mean([len(w) for w in tok])) if tok else 0.0
    )
    df["text_std_len"]    = raw_tokens.apply(
        lambda tok: float(np.std([len(w) for w in tok])) if tok else 0.0
    )

    # reading time (250 wpm) and lexical diversity
    df["reading_time_min"] = (df["word_count"] / 250).round(2)
    df["lexical_density"]  = raw_tokens.apply(
        lambda tok: round(len(set(tok)) / len(tok), 4) if tok else 0.0
    )
    return df


def summarize_features(df):
    """Descriptive statistics computed directly with numpy on the feature matrix."""
    cols = ["word_count","sentence_count","avg_word_length",
            "reading_time_min","lexical_density","text_std_len"]
    cols = [c for c in cols if c in df.columns]
    M = df[cols].to_numpy(dtype=float)
    return pd.DataFrame({
        "feature": cols,
        "mean":    np.mean(M, axis=0).round(4),
        "std":     np.std(M,  axis=0).round(4),
        "min":     np.min(M,  axis=0).round(4),
        "median":  np.median(M, axis=0).round(4),
        "max":     np.max(M,  axis=0).round(4),
    }).set_index("feature")


ML Analysis with scikit-learn
1. **TF-IDF vectorization** (unigrams + bigrams)  
2. **Silhouette-score sweep** to find the best `k`  
3. **KMeans** topic clustering  
4. **Top keyword extraction** per cluster  


In [14]:
def build_tfidf_matrix(df, max_features=5000, ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=True,   # log-scale TF dampens high-frequency terms
        min_df=2,
        max_df=0.90,
    )
    matrix = vectorizer.fit_transform(df["clean_text"]).toarray()
    print(f"TF-IDF matrix: {matrix.shape[0]} articles × {matrix.shape[1]} features")
    return matrix, vectorizer


def find_optimal_clusters(matrix, k_min=2, k_max=8):
    #Sweep k and return the value with the highest silhouette score
    k_max = min(k_max, matrix.shape[0] - 1)
    scores = {}
    for k in range(k_min, k_max + 1):
        labels = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(matrix)
        scores[k] = silhouette_score(matrix, labels)
        print(f"  k={k}  silhouette={scores[k]:.4f}")
    best_k = max(scores, key=scores.get)
    print(f"→ Optimal k={best_k}  (silhouette={scores[best_k]:.4f})")
    return best_k


def cluster_articles(df, matrix, n_clusters):
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    df = df.copy()
    df["cluster"] = km.fit_predict(matrix)
    centroids = km.cluster_centers_[df["cluster"].values]
    df["centroid_distance"] = np.linalg.norm(matrix - centroids, axis=1).round(5)
    return df


def extract_cluster_keywords(df, vectorizer, matrix, top_n=8):
    #Average TF-IDF scores per cluster, return top-n terms with numpy argsort.
    feature_names = np.array(vectorizer.get_feature_names_out())
    keywords = {}
    for cid in sorted(df["cluster"].unique()):
        mask = (df["cluster"] == cid).values
        mean_scores = np.mean(matrix[mask], axis=0)
        top_idx = np.argsort(mean_scores)[::-1][:top_n]
        keywords[cid] = feature_names[top_idx].tolist()
    return keywords


def compute_similarity_matrix(matrix):
    #Pairwise cosine similarity after L2 normalisation.
    normed = normalize(matrix, norm="l2")
    return cosine_similarity(normed).round(4)


def find_similar_articles(df, sim_matrix, article_idx, top_n=3):
    sims = sim_matrix[article_idx].copy()
    sims[article_idx] = -1
    top_idx = np.argsort(sims)[::-1][:top_n]
    result = df.iloc[top_idx][["title", "cluster"]].copy()
    result["similarity"] = sims[top_idx].round(4)
    return result.reset_index(drop=True)


Configure URLs


In [33]:
URLS = [
    "https://www.foxnews.com/opinion/jonathan-turley-chief-justice-roberts-could-learn-baseball-great-ted-williams-comes-leaks",
    "https://www.cnn.com/2026/04/19/business/gas-prices-saving-tips",
    "https://finance.yahoo.com/economy/article/retire-to-new-york-city-new-index-redefines-what-makes-a-great-place-for-retirees-103000325.html",
    "https://apnews.com/article/china-congress-economy-growth-target-6459c5f5b9bd7a538f9d7a7f17c304ef",
    "https://apnews.com/article/humanoid-robots-half-marathon-beijing-302d0c4781bab20100d6a0bb4e77b629",
    "https://apnews.com/article/ukraine-russia-chernobyl-nature-rebounds-a0b252dc78a539947835acec8540b9fe",
    "https://apnews.com/article/cuba-oil-embargo-crisis-havana-nightlife-4b8f1da8acf1aa8cb5f6b425d85ff1a4",
]

Run

In [34]:
#1. scrape articles
df_raw = scrape_articles(URLS)
df_raw[["title", "author", "char_count"]].head()

[1/7] https://www.foxnews.com/opinion/jonathan-turley-chief-justice-roberts-could-learn-baseball-great-ted-williams-comes-leaks
[2/7] https://www.cnn.com/2026/04/19/business/gas-prices-saving-tips
[3/7] https://finance.yahoo.com/economy/article/retire-to-new-york-city-new-index-redefines-what-makes-a-great-place-for-retirees-103000325.html
[4/7] https://apnews.com/article/china-congress-economy-growth-target-6459c5f5b9bd7a538f9d7a7f17c304ef
[5/7] https://apnews.com/article/humanoid-robots-half-marathon-beijing-302d0c4781bab20100d6a0bb4e77b629
[6/7] https://apnews.com/article/ukraine-russia-chernobyl-nature-rebounds-a0b252dc78a539947835acec8540b9fe
[7/7] https://apnews.com/article/cuba-oil-embargo-crisis-havana-nightlife-4b8f1da8acf1aa8cb5f6b425d85ff1a4

✓ 7 usable articles scraped.


,title,author,char_count
0,JONATHAN TURLEY: Chief Justice Roberts could l...,Unknown,7400
1,Gas prices won’t fall quickly. Here are ways t...,Unknown,4525
2,Yahoo Finance,Unknown,6583
3,Key takeaways from China’s new 5-year economic...,Unknown,4946
4,A humanoid robot sprints to victory in Beijing...,Unknown,4606


In [35]:
#2. feature engineering
df = compute_features(df_raw)

display_cols = ["title", "word_count", "avg_word_length", "lexical_density", "reading_time_min"]
df[display_cols]

,title,word_count,avg_word_length,lexical_density,reading_time_min
0,JONATHAN TURLEY: Chief Justice Roberts could l...,1196,5.188127,0.5075,4.78
1,Gas prices won’t fall quickly. Here are ways t...,779,4.810013,0.5302,3.12
2,Yahoo Finance,1072,5.141791,0.5308,4.29
3,Key takeaways from China’s new 5-year economic...,782,5.326087,0.5358,3.13
4,A humanoid robot sprints to victory in Beijing...,731,5.302326,0.5171,2.92
5,Chernobyl’s radioactive landscape is testament...,882,5.646259,0.5125,3.53
6,An energy blockade on Cuba pulls the plug on H...,664,5.379518,0.6461,2.66


In [36]:
# 3. summarize
summarize_features(df)

,mean,std,min,median,max
feature,,,,,
word_count,872.2857,179.2721,664.0000,782.0000,1196.0000
sentence_count,50.7143,11.2595,35.0000,46.0000,67.0000
avg_word_length,5.2563,0.2364,4.8100,5.3023,5.6463
reading_time_min,3.4900,0.7157,2.6600,3.1300,4.7800
lexical_density,0.5400,0.0444,0.5075,0.5302,0.6461
text_std_len,2.8792,0.1837,2.5008,2.8862,3.0984


In [37]:
#4. TF-IDF Vectorization with scikit-learn
tfidf_matrix, vectorizer = build_tfidf_matrix(df, max_features=5000)

TF-IDF matrix: 7 articles × 383 features


In [38]:
#5. optimal cluster count
n_clusters = find_optimal_clusters(tfidf_matrix, k_min=2, k_max=min(8, len(df) - 1))

  k=2  silhouette=0.0133
  k=3  silhouette=0.0210
  k=4  silhouette=0.0633
  k=5  silhouette=0.0395
  k=6  silhouette=0.0349
→ Optimal k=4  (silhouette=0.0633)


In [39]:
#6. plug in optimal k for kmeans clustering
print(f"optimal k={n_clusters}")
df = cluster_articles(df, tfidf_matrix, n_clusters)
df[["title", "cluster", "centroid_distance"]]

optimal k=4


,title,cluster,centroid_distance
0,JONATHAN TURLEY: Chief Justice Roberts could l...,2,0.00000
1,Gas prices won’t fall quickly. Here are ways t...,0,0.58531
2,Yahoo Finance,0,0.58531
3,Key takeaways from China’s new 5-year economic...,1,0.54023
4,A humanoid robot sprints to victory in Beijing...,1,0.54023
5,Chernobyl’s radioactive landscape is testament...,3,0.58808
6,An energy blockade on Cuba pulls the plug on H...,3,0.58808


In [40]:
#7. Cluster Keywords
cluster_keywords = extract_cluster_keywords(df, vectorizer, tfidf_matrix, top_n=8)

for cid, kws in cluster_keywords.items():
    members = df[df["cluster"] == cid]["title"].tolist()
    print(f"\nCluster {cid}  ({len(members)} articles)")
    print(f"  Keywords : {', '.join(kws)}")
    for t in members:
        print(f"  • {t}")


Cluster 0  (2 articles)
  Keywords : can, off, said, certain, every, cost, financial, higher
  • Gas prices won’t fall quickly. Here are ways to pay less at the pump right now
  • Yahoo Finance

Cluster 1  (2 articles)
  Keywords : beijing, china, chinese, wong, year, last year, plan, target
  • Key takeaways from China’s new 5-year economic blueprint and growth target
  • A humanoid robot sprints to victory in Beijing, beating the human half-marathon world record

Cluster 2  (1 articles)
  Keywords : chief, trump, rules, congress, donald, see, both, american
  • JONATHAN TURLEY: Chief Justice Roberts could learn from baseball great Ted Williams when it comes to leaks

Cluster 3  (2 articles)
  Keywords : wednesday, wednesday april, april, inside, small, life, said, today
  • Chernobyl’s radioactive landscape is testament to nature’s resilience and survival spirit
  • An energy blockade on Cuba pulls the plug on Havana’s legendary nightlife


Export Results

In [41]:
import os
os.makedirs("output", exist_ok=True)

# Enriched article data
export_cols = ["title","author","url","cluster","centroid_distance",
               "word_count","sentence_count","avg_word_length",
               "reading_time_min","lexical_density","text_std_len"]
df[[c for c in export_cols if c in df.columns]].to_csv("output/articles_analyzed.csv", index=False)

# Cluster keyword report
with open("output/cluster_report.txt", "w") as f:
    f.write("CLUSTER REPORT\n" + "="*55 + "\n\n")
    for cid, kws in sorted(cluster_keywords.items()):
        members = df[df["cluster"] == cid]["title"].tolist()
        f.write(f"CLUSTER {cid}\n")
        f.write(f"  Keywords: {', '.join(kws)}\n")
        for t in members:
            f.write(f"  • {t}\n")
        f.write("\n")

print("Saved to output/:")
print("  articles_analyzed.csv")
print("  similarity_matrix.csv")
print("  cluster_report.txt")

Saved to output/:
  articles_analyzed.csv
  similarity_matrix.csv
  cluster_report.txt
